# 05 — Deployment-matched RUL validation and controlled XGBoost ablation

This notebook follows the full-dataset robustness experiment. It is designed to investigate why a model selected under fixed multi-horizon validation did not transfer well to the official last-observed-cycle evaluation.

## What is compared

- Training RUL caps: **125, 175, 200 cycles**.
- Causal sensor-history windows: **5 and 10 cycles**.
- Feature modes: **baseline** and **operating-regime-aware**.
- The same previously selected XGBoost hyperparameters are held fixed across the 12 candidates so the ablation primarily compares target/feature choices.

## Validation design and leakage guardrails

- Whole engines are split by subset into development (60%), validation (20%) and locked audit (20%).
- Pseudo-cutoff cycles are sampled from the distribution of **observed terminal cycle counts** in the unlabeled test trajectories. Only `test_FD00X.txt` files are loaded; this notebook does **not** open any `RUL_FD00X.txt` labels.
- Sensor features use current and preceding observations only. `engine_key` is an identifier and never enters the model matrix.
- Any K-means operating-regime transformer and its sensor statistics are fit only on the engine partition allowed for that stage.
- Select on validation; use the locked audit once after selection; then refit the chosen configuration on all labelled training trajectories.

This is a deployment-oriented internal validation strategy, not proof of real-world industrial readiness. NASA C-MAPSS is simulated benchmark data. The official test labels have been inspected in earlier project iterations, so future official-test results should be described as exploratory benchmark comparisons, not as a pristine blind test.

In [ ]:
from pathlib import Path
import json
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
SENSOR_COLS = [f"s_{i}" for i in range(1, 22)]
SETTING_COLS = ["setting_1", "setting_2", "setting_3"]
WINDOW_CANDIDATES = [5, 10]
TARGET_CAPS = [125, 175, 200]
FEATURE_MODES = ["baseline", "regime_aware"]
N_REGIMES = 6
N_CUTOFFS_PER_ENGINE = 3

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CMAPSS_DIR = PROJECT_ROOT / "CMAPSSData"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in (RAW_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def find_dataset_file(filename):
    for path in (RAW_DIR / filename, CMAPSS_DIR / filename, PROJECT_ROOT / filename):
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing {filename}; checked data/raw/, CMAPSSData/, and project root.")


def load_cmapss_file(filename):
    frame = pd.read_csv(find_dataset_file(filename), sep=r"\s+", header=None, names=COLUMN_NAMES, engine="python")
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"Unexpected schema for {filename}: {frame.shape}")
    return frame


def load_all_training():
    parts = []
    for subset in SUBSETS:
        frame = load_cmapss_file(f"train_{subset}.txt")
        frame["subset"] = subset
        frame["engine_key"] = subset + "_" + frame["unit_nr"].astype(int).astype(str).str.zfill(3)
        parts.append(frame)
    return pd.concat(parts, ignore_index=True)


def load_unlabeled_test_terminal_cycles():
    """Read test sensor files only. Deliberately does not load official RUL_*.txt labels."""
    ages = {}
    summary = []
    for subset in SUBSETS:
        frame = load_cmapss_file(f"test_{subset}.txt")
        terminal = frame.groupby("unit_nr")["time_cycles"].max().astype(int).to_numpy()
        ages[subset] = terminal
        summary.append({"subset": subset, "test_engines": int(len(terminal)),
                        "min_observed_terminal_cycle": int(terminal.min()),
                        "median_observed_terminal_cycle": float(np.median(terminal)),
                        "max_observed_terminal_cycle": int(terminal.max())})
    return ages, pd.DataFrame(summary)

train_raw = load_all_training()
observed_test_ages, observed_age_summary = load_unlabeled_test_terminal_cycles()
print(f"Training rows: {len(train_raw):,} | training engines: {train_raw['engine_key'].nunique():,}")
print("Observed test trajectory lengths loaded from test sensor files only (no official RUL labels opened):")
display(observed_age_summary)


## 1. Make causal features and a fixed engine-level split

The target is raw remaining life (`last_cycle - current_cycle`); target caps are applied separately at model fitting time. All candidate feature matrices retain the same row ordering, so pseudo-cutoff row indices can be reused across windows.

In [ ]:
def engineer_base_features(frame, window):
    ordered = frame.sort_values(["subset", "unit_nr", "time_cycles"]).reset_index(drop=True).copy()
    required = ["subset", "unit_nr", "engine_key", "time_cycles"] + SETTING_COLS + SENSOR_COLS
    features = ordered[required].copy()
    groups = ordered.groupby(["subset", "unit_nr"], sort=False)
    for sensor in SENSOR_COLS:
        signal = groups[sensor]
        features[f"{sensor}_roll_mean_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=1).mean())
        features[f"{sensor}_roll_std_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=2).std(ddof=0)).fillna(0.0)
        features[f"{sensor}_delta_1"] = signal.diff().fillna(0.0)
    for subset in SUBSETS:
        features[f"subset_{subset}"] = (ordered["subset"] == subset).astype(np.int8)
    final_cycle = ordered.groupby("engine_key")["time_cycles"].max()
    features["RUL_true"] = final_cycle.reindex(ordered["engine_key"]).to_numpy() - ordered["time_cycles"].to_numpy()
    return features

ID_COLS = {"subset", "unit_nr", "engine_key", "RUL_true"}
BASE_FEATURES_BY_WINDOW = {w: engineer_base_features(train_raw, w) for w in WINDOW_CANDIDATES}
BASE_COLUMNS_BY_WINDOW = {w: [c for c in BASE_FEATURES_BY_WINDOW[w].columns if c not in ID_COLS] for w in WINDOW_CANDIDATES}
for w in WINDOW_CANDIDATES:
    assert BASE_FEATURES_BY_WINDOW[w][BASE_COLUMNS_BY_WINDOW[w]].notna().all().all()
    assert "engine_key" not in BASE_COLUMNS_BY_WINDOW[w] and "unit_nr" not in BASE_COLUMNS_BY_WINDOW[w]

# Same deterministic 60/20/20 split policy as the prior robustness notebook.
DEV_KEYS, VAL_KEYS, AUDIT_KEYS = [], [], []
for subset in SUBSETS:
    keys = np.sort(train_raw.loc[train_raw["subset"].eq(subset), "engine_key"].unique())
    dev_keys, remainder = train_test_split(keys, test_size=0.40, random_state=RANDOM_STATE)
    val_keys, audit_keys = train_test_split(remainder, test_size=0.50, random_state=RANDOM_STATE + 1)
    DEV_KEYS.extend(dev_keys.tolist())
    VAL_KEYS.extend(val_keys.tolist())
    AUDIT_KEYS.extend(audit_keys.tolist())

assert set(DEV_KEYS).isdisjoint(VAL_KEYS)
assert set(DEV_KEYS).isdisjoint(AUDIT_KEYS)
assert set(VAL_KEYS).isdisjoint(AUDIT_KEYS)

display(pd.DataFrame([
    {"subset": s,
     "development_engines": len(set(DEV_KEYS) & set(train_raw.loc[train_raw.subset.eq(s), "engine_key"])),
     "validation_engines": len(set(VAL_KEYS) & set(train_raw.loc[train_raw.subset.eq(s), "engine_key"])),
     "audit_engines": len(set(AUDIT_KEYS) & set(train_raw.loc[train_raw.subset.eq(s), "engine_key"]))}
    for s in SUBSETS
]))
print(f"Split totals: dev={len(DEV_KEYS)}, validation={len(VAL_KEYS)}, locked audit={len(AUDIT_KEYS)}")


## 2. Build deployment-matched pseudo-cutoff cases

For each held-out training engine, sample three observed-age values from the corresponding subset's *unlabeled test trajectory end-cycle distribution*. If a sampled age exceeds the held-out engine's complete life, it is clipped to the last pre-failure cycle. The hidden future length in that held-out training trajectory supplies the validation truth. This does not use official test RUL labels.

In [ ]:
def build_age_matched_points(features, engine_keys, age_by_subset, n_per_engine=3, seed=420):
    rng = np.random.default_rng(seed)
    allowed = set(engine_keys)
    records = []
    use = features.loc[features["engine_key"].isin(allowed)]
    for engine_key, group in use.groupby("engine_key", sort=True):
        group = group.sort_values("time_cycles")
        subset = str(group["subset"].iloc[0])
        last_cycle = int(group["time_cycles"].max())
        empirical_ages = np.asarray(age_by_subset[subset], dtype=int)
        for rep in range(n_per_engine):
            sampled_age = int(rng.choice(empirical_ages))
            cutoff = max(1, min(sampled_age, last_cycle - 1))
            candidates = group.loc[group["time_cycles"].le(cutoff)]
            if candidates.empty:
                continue
            row = candidates.iloc[-1]
            actual_cutoff = int(row["time_cycles"])
            records.append({
                "base_row_idx": int(row.name),
                "subset": subset,
                "engine_key": str(engine_key),
                "unit_nr": int(row["unit_nr"]),
                "cutoff_cycle": actual_cutoff,
                "sampled_test_age": sampled_age,
                "true_rul_cycles": float(last_cycle - actual_cutoff),
                "replicate": int(rep),
            })
    result = pd.DataFrame(records)
    if result.empty:
        raise ValueError("No pseudo-cutoff cases were created.")
    return result

# Row positions are stable across window variants because sort/order is identical.
val_points = build_age_matched_points(BASE_FEATURES_BY_WINDOW[5], VAL_KEYS, observed_test_ages, N_CUTOFFS_PER_ENGINE, seed=420)
audit_points = build_age_matched_points(BASE_FEATURES_BY_WINDOW[5], AUDIT_KEYS, observed_test_ages, N_CUTOFFS_PER_ENGINE, seed=421)
print("Validation cases:", len(val_points), "across", val_points.engine_key.nunique(), "held-out engines")
print("Locked audit cases:", len(audit_points), "across", audit_points.engine_key.nunique(), "held-out engines")
display(val_points.groupby("subset").agg(cases=("engine_key", "size"), engines=("engine_key", "nunique"), median_true_rul=("true_rul_cycles", "median"), max_true_rul=("true_rul_cycles", "max")).reset_index())


## 3. Operating-regime feature transformer

The regime-aware mode clusters the three operating settings using a scaler and K-means fitted only on allowed training engines. It then adds one-hot regime indicators and within-subset/regime standardized versions of sensor values and rolling means. The baseline remains the original feature set. This is an experiment, not an assumption that normalization must improve performance.

In [ ]:
def make_stats(part, source_columns):
    stats = {}
    for col in source_columns:
        vals = pd.to_numeric(part[col], errors="coerce").to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            stats[col] = (0.0, 1.0)
        else:
            stats[col] = (float(np.mean(vals)), max(float(np.std(vals, ddof=0)), 1e-6))
    return stats


def fit_regime_transformer(features, fit_engine_keys, window, random_state=RANDOM_STATE):
    source_columns = SENSOR_COLS + [f"{s}_roll_mean_{window}" for s in SENSOR_COLS]
    fit = features.loc[features["engine_key"].isin(set(fit_engine_keys))].copy().reset_index(drop=True)
    if fit.empty:
        raise ValueError("No rows supplied to fit regime transformer.")
    scaler = StandardScaler().fit(fit[SETTING_COLS])
    km = KMeans(n_clusters=N_REGIMES, n_init=10, random_state=random_state)
    fit["_regime"] = km.fit_predict(scaler.transform(fit[SETTING_COLS])).astype(int)
    group_stats = {}
    for (subset, regime), group in fit.groupby(["subset", "_regime"], sort=True):
        group_stats[(str(subset), int(regime))] = make_stats(group, source_columns)
    subset_stats = {str(s): make_stats(g, source_columns) for s, g in fit.groupby("subset", sort=True)}
    global_stats = make_stats(fit, source_columns)
    return {"setting_scaler": scaler, "kmeans": km, "group_stats": group_stats,
            "subset_stats": subset_stats, "global_stats": global_stats,
            "n_regimes": N_REGIMES, "source_columns": source_columns,
            "rolling_window": int(window),
            "method": "KMeans on standardized operating settings; sensor/rolling-mean z-scores by subset and regime"}


def add_regime_features(features, transformer):
    out = features.copy().reset_index(drop=True)
    regime = transformer["kmeans"].predict(transformer["setting_scaler"].transform(out[SETTING_COLS])).astype(int)
    out["_regime"] = regime
    for r in range(int(transformer["n_regimes"])):
        out[f"operating_regime_{r}"] = (regime == r).astype(np.int8)
    group_positions = {}
    for pos, key in enumerate(zip(out["subset"].astype(str), regime)):
        group_positions.setdefault((key[0], int(key[1])), []).append(pos)
    for source_col in transformer["source_columns"]:
        means = np.empty(len(out), dtype=float)
        stds = np.empty(len(out), dtype=float)
        for key, positions in group_positions.items():
            stats = transformer["group_stats"].get(key)
            if stats is None:
                stats = transformer["subset_stats"].get(key[0], transformer["global_stats"])
            mean, sd = stats.get(source_col, transformer["global_stats"][source_col])
            means[positions] = mean
            stds[positions] = max(float(sd), 1e-6)
        out[f"{source_col}_regime_z"] = (out[source_col].to_numpy(dtype=float) - means) / stds
    out.drop(columns=["_regime"], inplace=True)
    return out

REGIME_COLUMNS_BY_WINDOW = {
    w: [f"operating_regime_{r}" for r in range(N_REGIMES)] +
       [f"{c}_regime_z" for c in SENSOR_COLS + [f"{s}_roll_mean_{w}" for s in SENSOR_COLS]]
    for w in WINDOW_CANDIDATES
}


## 4. Controlled candidate comparison

The same fixed XGBoost settings are used for each candidate to reduce confounding from hyperparameter changes. They come from the earlier grouped-CV search; this sweep tests feature mode, target cap and window length. The selection score is the mean per-case asymmetric penalty averaged equally across the four subsets; macro-subset MAE is the tie-breaker. This objective is evaluated on the age-matched validation cases only.

In [ ]:
# Fixed settings from the earlier full-dataset grouped-CV baseline search.
# Keep these fixed for the ablation so cap/window/feature effects are more interpretable.
FIXED_XGB_PARAMS = {
    "n_estimators": 550,
    "max_depth": 5,
    "learning_rate": 0.03,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "min_child_weight": 5,
    "reg_lambda": 7.0,
}


def capped_target(y, cap):
    y = np.asarray(y, dtype=float)
    return y if cap is None else np.minimum(y, float(cap))


def nasa_penalty_array(y_true, y_pred):
    error = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    return np.where(error < 0,
                    np.exp(np.clip(-error / 13.0, 0, 50)) - 1,
                    np.exp(np.clip(error / 10.0, 0, 50)) - 1)


def metric_pack(points, prediction):
    scored = points[["subset", "engine_key", "true_rul_cycles"]].copy().reset_index(drop=True)
    scored["predicted_rul_cycles"] = np.maximum(0.0, np.asarray(prediction, dtype=float))
    scored["error_pred_minus_true"] = scored["predicted_rul_cycles"] - scored["true_rul_cycles"]
    scored["absolute_error"] = scored["error_pred_minus_true"].abs()
    scored["nasa_penalty"] = nasa_penalty_array(scored["true_rul_cycles"], scored["predicted_rul_cycles"])
    by_subset = scored.groupby("subset").agg(
        cases=("engine_key", "size"),
        mae=("absolute_error", "mean"),
        rmse=("error_pred_minus_true", lambda s: float(np.sqrt(np.mean(np.square(s))))),
        nasa_score_per_case=("nasa_penalty", "mean"),
        late_pct=("error_pred_minus_true", lambda s: float(np.mean(s > 0) * 100)),
        mean_bias=("error_pred_minus_true", "mean"),
    ).reset_index()
    summary = {
        "pooled_mae": float(scored["absolute_error"].mean()),
        "pooled_rmse": float(np.sqrt(np.mean(scored["error_pred_minus_true"] ** 2))),
        "pooled_r2": float(r2_score(scored["true_rul_cycles"], scored["predicted_rul_cycles"])),
        "macro_subset_mae": float(by_subset["mae"].mean()),
        "macro_subset_nasa_score_per_case": float(by_subset["nasa_score_per_case"].mean()),
        "macro_subset_late_pct": float(by_subset["late_pct"].mean()),
        "macro_subset_mean_bias": float(by_subset["mean_bias"].mean()),
    }
    return summary, by_subset, scored


def feature_view(window, mode, fit_engine_keys):
    base_features = BASE_FEATURES_BY_WINDOW[window]
    base_cols = BASE_COLUMNS_BY_WINDOW[window]
    if mode == "baseline":
        return base_features, base_cols, None
    transformer = fit_regime_transformer(base_features, fit_engine_keys, window=window)
    augmented = add_regime_features(base_features, transformer)
    return augmented, base_cols + REGIME_COLUMNS_BY_WINDOW[window], transformer

candidate_rows = []
candidate_predictions = {}
for window in WINDOW_CANDIDATES:
    base_features = BASE_FEATURES_BY_WINDOW[window]
    for mode in FEATURE_MODES:
        # Fit the feature transformer on development engines only.
        feature_frame, feature_cols, transformer = feature_view(window, mode, DEV_KEYS)
        dev_mask = feature_frame["engine_key"].isin(DEV_KEYS)
        X_dev = feature_frame.loc[dev_mask, feature_cols]
        y_dev_raw = feature_frame.loc[dev_mask, "RUL_true"].to_numpy(dtype=float)
        val_X = feature_frame.iloc[val_points["base_row_idx"].to_numpy(dtype=int)][feature_cols]
        for cap in TARGET_CAPS:
            model = XGBRegressor(**FIXED_XGB_PARAMS, objective="reg:squarederror", tree_method="hist", random_state=RANDOM_STATE, n_jobs=4)
            model.fit(X_dev, capped_target(y_dev_raw, cap))
            pred = np.maximum(0.0, model.predict(val_X))
            summary, by_subset, scored = metric_pack(val_points, pred)
            candidate_id = f"{mode}_w{window}_cap{cap}"
            candidate_predictions[candidate_id] = scored.assign(candidate_id=candidate_id)
            candidate_rows.append({"candidate_id": candidate_id, "feature_mode": mode, "rolling_window": window, "target_cap": cap, **summary})
            print(f"Finished {candidate_id}: macro NASA={summary['macro_subset_nasa_score_per_case']:.3f}, macro MAE={summary['macro_subset_mae']:.2f}")

comparison = pd.DataFrame(candidate_rows).sort_values(
    ["macro_subset_nasa_score_per_case", "macro_subset_mae"], ascending=True
).reset_index(drop=True)
comparison.to_csv(REPORTS_DIR / "deployment_matched_candidate_comparison.csv", index=False)
display(comparison.round(3))

WINNER = comparison.iloc[0].to_dict()
WINNER_MODE = str(WINNER["feature_mode"])
WINNER_WINDOW = int(WINNER["rolling_window"])
WINNER_CAP = int(WINNER["target_cap"])
WINNER_ID = str(WINNER["candidate_id"])
print("Validation-selected candidate:", WINNER_ID)
print("Selection was based only on held-out training engines and unlabeled test-cycle-length distribution; official RUL labels were not opened.")


## 5. Inspect candidate performance by subset and RUL range

This breakdown is a diagnostic tool for the validation partition. Do not use the official RUL test labels here. Look for whether a candidate's average score is hiding large errors on FD002/FD004 or in the high-RUL band.

In [ ]:
winner_scored = candidate_predictions[WINNER_ID].copy()
winner_scored["rul_band"] = pd.cut(winner_scored["true_rul_cycles"], bins=[0, 30, 60, 125, 180, np.inf], labels=["1-30", "31-60", "61-125", "126-180", "181+"], include_lowest=True)
val_band_report = winner_scored.groupby(["subset", "rul_band"], observed=False).agg(
    cases=("engine_key", "size"),
    mae=("absolute_error", "mean"),
    rmse=("error_pred_minus_true", lambda s: float(np.sqrt(np.mean(np.square(s))))),
    mean_bias=("error_pred_minus_true", "mean"),
    late_pct=("error_pred_minus_true", lambda s: float(np.mean(s > 0) * 100)),
).reset_index()
val_band_report.to_csv(REPORTS_DIR / "deployment_matched_validation_by_subset_band.csv", index=False)
display(val_band_report.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(winner_scored["true_rul_cycles"], winner_scored["predicted_rul_cycles"], alpha=0.6)
lim = float(max(winner_scored["true_rul_cycles"].max(), winner_scored["predicted_rul_cycles"].max(), 1))
axes[0].plot([0, lim], [0, lim], "--", color="gray")
axes[0].set(title=f"Validation: {WINNER_ID}", xlabel="True RUL (cycles)", ylabel="Predicted RUL (cycles)")
axes[1].hist(winner_scored["error_pred_minus_true"], bins=25, edgecolor="white")
axes[1].axvline(0, linestyle="--", color="gray")
axes[1].set(title="Validation error distribution", xlabel="Predicted − true RUL (cycles)", ylabel="Pseudo-cutoff cases")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "deployment_matched_validation_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Evaluate the selected candidate once on the locked audit engines

The audit is not used to choose between candidate configurations. The model is fit on development + validation engines using the validation-selected cap/window/mode and the fixed hyperparameters above, then evaluated on the locked audit engines with the same age-matched cutoff protocol.

In [ ]:
fit_keys_audit = sorted(set(DEV_KEYS + VAL_KEYS))
audit_feature_frame, audit_feature_cols, audit_transformer = feature_view(WINNER_WINDOW, WINNER_MODE, fit_keys_audit)
fit_mask = audit_feature_frame["engine_key"].isin(fit_keys_audit)
fit_x = audit_feature_frame.loc[fit_mask, audit_feature_cols]
fit_y = capped_target(audit_feature_frame.loc[fit_mask, "RUL_true"].to_numpy(dtype=float), WINNER_CAP)

# Rebuild the audit point rows using the same seed independent of the selected window.
# Row indexing is identical for all windows, and the audit engines were not used in selection.
audit_model = XGBRegressor(**FIXED_XGB_PARAMS, objective="reg:squarederror", tree_method="hist", random_state=RANDOM_STATE, n_jobs=4)
audit_model.fit(fit_x, fit_y)
audit_X = audit_feature_frame.iloc[audit_points["base_row_idx"].to_numpy(dtype=int)][audit_feature_cols]
audit_pred = np.maximum(0.0, audit_model.predict(audit_X))
audit_summary_metrics, audit_by_subset, audit_scored = metric_pack(audit_points, audit_pred)
audit_scored = pd.concat([audit_points[["subset", "engine_key", "unit_nr", "cutoff_cycle", "sampled_test_age", "true_rul_cycles"]].reset_index(drop=True), audit_scored[["predicted_rul_cycles", "error_pred_minus_true", "absolute_error", "nasa_penalty"]]], axis=1)
audit_scored.to_csv(REPORTS_DIR / "deployment_matched_locked_audit_predictions.csv", index=False)
audit_by_subset.to_csv(REPORTS_DIR / "deployment_matched_locked_audit_by_subset.csv", index=False)

# Stratified engine bootstrap: resample whole engines within each subset, not individual rows.
per_engine = audit_scored.groupby(["subset", "engine_key"]).agg(engine_mae=("absolute_error", "mean")).reset_index()
rng = np.random.default_rng(RANDOM_STATE + 500)
boot = []
for _ in range(1500):
    subset_means = []
    for subset in SUBSETS:
        vals = per_engine.loc[per_engine.subset.eq(subset), "engine_mae"].to_numpy(dtype=float)
        if len(vals):
            subset_means.append(float(rng.choice(vals, size=len(vals), replace=True).mean()))
    boot.append(float(np.mean(subset_means)))
mae_ci = np.quantile(boot, [0.025, 0.975])

audit_report = {
    "validation_selected_candidate": WINNER_ID,
    "feature_mode": WINNER_MODE,
    "rolling_window": WINNER_WINDOW,
    "target_cap": WINNER_CAP,
    "fixed_xgboost_params": FIXED_XGB_PARAMS,
    "audit_engines": int(per_engine["engine_key"].nunique()),
    "audit_cases": int(len(audit_scored)),
    "metrics": audit_summary_metrics,
    "stratified_engine_bootstrap_95pct_interval_macro_subset_engine_mae": [float(mae_ci[0]), float(mae_ci[1])],
    "per_subset_metrics": audit_by_subset.to_dict(orient="records"),
    "guardrails": {
        "official_rul_label_files_loaded": False,
        "audit_used_for_candidate_selection": False,
        "engine_level_split": True,
        "feature_transformer_fit_only_on_dev_plus_validation_engines": True,
    },
    "interpretation": "Internal age-matched pseudo-cutoff audit, not official NASA test performance and not real-world industrial validation."
}
(REPORTS_DIR / "deployment_matched_locked_audit_summary.json").write_text(json.dumps(audit_report, indent=2))
print(json.dumps(audit_report, indent=2))
display(audit_by_subset.round(3))


## 7. Refit the chosen configuration on all training engines and save an artifact

The selected target cap/window/mode comes only from validation. The locked audit is reported above but does not alter the selection. Before replacing the main artifact, the existing robustness-selected model is copied to a snapshot file if one exists.

In [ ]:
# Preserve the currently active robustness-selected artifact before writing the next candidate.
MODEL_PATH = MODELS_DIR / "xgb_rul_model_all_subsets.joblib"
BACKUP_PATH = MODELS_DIR / "xgb_rul_model_robustness_selected_snapshot.joblib"
if MODEL_PATH.exists() and not BACKUP_PATH.exists():
    shutil.copy2(MODEL_PATH, BACKUP_PATH)
    print("Previous model snapshot saved to:", BACKUP_PATH)

final_base = BASE_FEATURES_BY_WINDOW[WINNER_WINDOW]
all_engine_keys = sorted(final_base["engine_key"].unique().tolist())
if WINNER_MODE == "regime_aware":
    final_transformer = fit_regime_transformer(final_base, all_engine_keys, window=WINNER_WINDOW)
    final_features = add_regime_features(final_base, final_transformer)
    final_feature_columns = BASE_COLUMNS_BY_WINDOW[WINNER_WINDOW] + REGIME_COLUMNS_BY_WINDOW[WINNER_WINDOW]
else:
    final_transformer = None
    final_features = final_base
    final_feature_columns = BASE_COLUMNS_BY_WINDOW[WINNER_WINDOW]

final_model = XGBRegressor(**FIXED_XGB_PARAMS, objective="reg:squarederror", tree_method="hist", random_state=RANDOM_STATE, n_jobs=4)
final_model.fit(final_features[final_feature_columns], capped_target(final_features["RUL_true"].to_numpy(dtype=float), WINNER_CAP))

artifact = {
    "model": final_model,
    "sensor_cols": SENSOR_COLS,
    "setting_cols": SETTING_COLS,
    "feature_columns": final_feature_columns,
    "base_feature_columns": BASE_COLUMNS_BY_WINDOW[WINNER_WINDOW],
    "rolling_window": int(WINNER_WINDOW),
    "rul_cap": int(WINNER_CAP),
    "best_params": FIXED_XGB_PARAMS,
    "dataset": "NASA PCoE C-MAPSS FD001-FD004 pooled XGBoost model (deployment-matched validation-selected)",
    "subsets": SUBSETS,
    "target_unit": "operating cycles",
    "random_state": RANDOM_STATE,
    "n_training_rows": int(len(train_raw)),
    "n_training_engines": int(train_raw["engine_key"].nunique()),
    "n_model_features": int(len(final_feature_columns)),
    "feature_mode": WINNER_MODE,
    "regime_transformer": final_transformer,
    "experiment_selection": {
        "candidate_id": WINNER_ID,
        "target_cap": int(WINNER_CAP),
        "rolling_window": int(WINNER_WINDOW),
        "feature_mode": WINNER_MODE,
        "selection_metric": "age-matched pseudo-cutoff macro-subset asymmetric penalty; macro-subset MAE tie-breaker",
        "validation_rul_label_files_opened": False,
        "locked_audit_used_for_selection": False,
        "official_test_labels_used_for_fit_or_selection": False,
        "note": "Official test labels have been inspected in previous project iterations; report subsequent official-test metrics transparently as exploratory benchmark comparisons."
    },
    "target_policy": "Training RUL target cap selected on engine-held-out age-matched pseudo-cutoffs; official test RUL labels excluded from fitting and selection."
}
joblib.dump(artifact, MODEL_PATH)
final_summary = {
    "model_artifact": str(MODEL_PATH),
    "previous_model_snapshot": str(BACKUP_PATH) if BACKUP_PATH.exists() else None,
    "dataset": artifact["dataset"],
    "training_rows": artifact["n_training_rows"],
    "training_engines": artifact["n_training_engines"],
    "feature_count": artifact["n_model_features"],
    "selected_configuration": {"candidate_id": WINNER_ID, "feature_mode": WINNER_MODE, "rolling_window": WINNER_WINDOW, "target_cap": WINNER_CAP},
    "validation_metrics": WINNER,
    "locked_audit_summary_file": str(REPORTS_DIR / "deployment_matched_locked_audit_summary.json"),
    "official_test_labels_used_for_selection": False,
}
(REPORTS_DIR / "deployment_matched_final_training_summary.json").write_text(json.dumps(final_summary, indent=2))
print("Final model saved:", MODEL_PATH)
print("Selected configuration:", json.dumps(final_summary["selected_configuration"], indent=2))
print("Feature count:", len(final_feature_columns))
print("Next: run notebooks/03_test_evaluation_and_explainability.ipynb once for exploratory benchmark reporting, then restart the dashboard.")


## Interview interpretation checklist

- Why the original multi-horizon validation may not represent the final observed-cycle distribution.
- Why we use test **cycle lengths only**, never official test RUL labels, to choose pseudo-cutoff ages.
- Why target-cap, rolling-window and regime-feature effects are compared under fixed model hyperparameters.
- Why validation selects the configuration and the locked audit is reported once without selection.
- Why official test metrics can still be explored for continuity but, after earlier inspection, must not be described as an untouched blind result.
- Why simulated C-MAPSS benchmark results do not certify a real industrial or aviation maintenance system.